<a href="https://colab.research.google.com/github/AbdallaYoussef006/FlyRank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This notebook audits two things: the published FlyRank research paper, and my own model.
The question in both cases is the same — **does the validation design carry the claim being made?**

Simple words, honest numbers.


## 1. Two paper findings + my methodology questions

I read `docs/flyrank-seo-research-march-2026.pdf` (*The State of AI-Driven SEO*, March 2026).
Both findings below are interesting and both are honestly flagged by the authors themselves.
My questions are about how far each number can be pushed, not about whether the work was done well.

---

### Finding A — "The Freshness Multiplier" (Finding #4, p.9)

**The claim.** The growth-to-decline ratio by freshness window is reported as 7.88:1 at 31–90 days,
3.13:1 at 181–360 days, and **283:1 at 361+ days**. The headline reading is that recent freshness is
a strong growth lever.

**Where the label comes from.** "Growing" and "declining" come from `trend_direction`, which the paper
defines (p.5) as 30-day vs previous-30-day impression change: up is more than 10% growth, down is more
than 10% decline. So the label is a *short-window impressions delta*, not an editorial judgement.

**My questions.**

1. **The 283:1 ratio has a denominator of 1.** The paper states there are 283 growing pages and exactly
   one declining page in that bucket. A ratio whose denominator is a single page is not a measurement —
   one more declining page would halve it to 141:1. The authors do flag this in the chart read, and they
   correctly refuse to headline it. My question is whether it belongs on the chart at all: a reader who
   skims the bars sees a bar 35x taller than every other bar. A minimum-count rule (the paper sets one at
   n=50 per bucket on p.5) would have removed it, and that rule appears not to have been applied here.

2. **The refresh result has no control group.** The paper reports that 365+ day content refreshed within
   30 days shows a 3.2x health boost and 57x more impressions. But pages are not refreshed at random —
   an editor chooses them, and they choose pages they believe are worth saving. The comparison group
   (unrefreshed old pages) therefore contains everything nobody thought was worth the effort. That gap
   would appear even if refreshing did nothing. To carry a causal reading you would need refreshed and
   unrefreshed pages matched on prior impressions, position, and age — or better, a staggered rollout.
   The paper's own thesis says "refresh strong content before it decays," which is a *selection* rule,
   so the selection effect and the treatment effect are pointing the same direction and cannot be
   separated from this design.

---

### Finding B — "What Predicts Health?" (ML Appendix, p.27)

**The claim.** A holdout-tested Random Forest predicting health score finds Average Position at 43%
importance, Impressions at 32%, and Scroll Depth at 15%.

**Where the label comes from.** Health Score is defined on p.5 as a composite:
impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).

**My question.** The target is **built out of the features**. Position and impressions together make up
60 of the 100 points in the score, and the model assigns them 75% of its importance. The model is
therefore rediscovering the formula, not discovering a pattern in search behaviour — and a holdout split
cannot detect this, because the relationship is just as true in the test set as in the training set.
This is target leakage in the strict sense, and the accuracy would look strong no matter how the data
were split.

To their credit the paper says exactly this in the chart read — "read this as model behavior, not as a
standalone optimization order." My question is narrower: given that the answer is arithmetic rather than
evidence, what does running the model add? A reader who does not reach the caveat comes away believing
position was *discovered* to matter most.

---

### Why this matters for my own work

Finding B is the same trap the internship's own data guide warns about with `trend_direction` and
`trend_pct` — a label derived from a column that then reappears as a feature. My leakage audit in
section 3 checks for exactly this shape in my own feature set.


In [1]:
# The two numbers that make each claim fragile, stated plainly.
import pandas as pd

audit = pd.DataFrame([
    {
        "finding": "A — Freshness Multiplier (p.9)",
        "headline": "361+ day bucket shows 283:1 growth-to-decline",
        "what_carries_it": "283 growing pages vs 1 declining page",
        "my_question": "Denominator = 1; paper's own n>=50 minimum (p.5) not applied to this bucket",
    },
    {
        "finding": "A — Refresh effect (p.9)",
        "headline": "Refreshed 365+ content: 3.2x health, 57x impressions",
        "what_carries_it": "Refreshed vs unrefreshed old pages, unmatched",
        "my_question": "Editors choose which pages to refresh -> selection effect not separable from treatment",
    },
    {
        "finding": "B — What Predicts Health? (p.27)",
        "headline": "Avg position 43% + impressions 32% = 75% of importance",
        "what_carries_it": "Random Forest predicting Health Score",
        "my_question": "Health Score = impressions(30)+position(30)+CTR(20)+scroll(20); target built from features",
    },
])

pd.set_option("display.max_colwidth", 78)
print(audit.to_string(index=False))


                         finding                                               headline                               what_carries_it                                                                                my_question
  A — Freshness Multiplier (p.9)          361+ day bucket shows 283:1 growth-to-decline         283 growing pages vs 1 declining page                Denominator = 1; paper's own n>=50 minimum (p.5) not applied to this bucket
        A — Refresh effect (p.9)   Refreshed 365+ content: 3.2x health, 57x impressions Refreshed vs unrefreshed old pages, unmatched     Editors choose which pages to refresh -> selection effect not separable from treatment
B — What Predicts Health? (p.27) Avg position 43% + impressions 32% = 75% of importance         Random Forest predicting Health Score Health Score = impressions(30)+position(30)+CTR(20)+scroll(20); target built from features


## 2. My model under an honest split (before/after)

My Week-5 model predicts whether a content item's impressions will fall to 70% or less of the current
month in the following month.

I test my split design in two directions at once:

- **Random split vs client-grouped split.** A random split lets the same client — and often the same
  page in a different month — sit on both sides. The grouped split holds out 14 whole clients, so the
  test set is clients the model has never seen.
- **No volume floor vs a 100-impression floor.** Without a floor, a page going from 4 impressions to 1
  counts as a decline. That is arithmetic on noise, and it inflates the dataset with rows nobody would
  ever act on. The floor is a sensitivity check: if the result only survives without it, the result is noise.

I report **AUC and precision@k** alongside accuracy and F1. My output is a *ranked review queue*, not a
yes/no decision, and accuracy at a 0.5 threshold measures a task I am not actually doing. Precision@k
answers the question a content editor actually asks: *of the top N pages you handed me, how many were
really declining?*


In [2]:
# --- Setup: connect to the warehouse -------------------------------------
# Token comes from a Colab Secret named HF_TOKEN. Never paste it in a cell:
# this repo is public.
%pip -q install duckdb huggingface_hub scikit-learn matplotlib

import os, json, warnings
import numpy as np
import pandas as pd
import duckdb
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Confirm the release matches the documented build before trusting anything.
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT}").df())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       rows      min_d      max_d
0  78835655 2025-01-27 2026-06-30


In [3]:
# --- Rebuild the monthly modelling frame ---------------------------------
# Grain: one row per content item per calendar month.
# Features come from the CURRENT month; the label comes from the NEXT month.
#
# Cached to /content/ (Colab local disk), NOT the repo — datasets must never
# enter git, and .gitignore + CI both block them.
CACHE = "/content/monthly_frame.parquet"

if os.path.exists(CACHE):
    frame = pd.read_parquet(CACHE)
    print("loaded from cache")
else:
    frame = con.sql(f"""
        WITH monthly AS (
            SELECT
                client_hash_id,
                content_hash_id,
                DATE_TRUNC('month', report_date)                  AS month,
                SUM(gsc_impressions)                              AS impressions,
                SUM(gsc_clicks)                                   AS clicks,
                -- NULLIF: gsc_avg_position = 0 means "no data", NOT rank zero.
                AVG(NULLIF(gsc_avg_position, 0))                  AS avg_position,
                SUM(scroll_events)                                AS scroll_events,
                SUM(sessions_ai)                                  AS sessions_ai
            FROM {FACT}
            GROUP BY 1, 2, 3
        ),
        with_next AS (
            SELECT *,
                   LEAD(impressions) OVER (
                       PARTITION BY client_hash_id, content_hash_id ORDER BY month
                   ) AS next_month_impressions
            FROM monthly
        )
        SELECT * FROM with_next
        WHERE next_month_impressions IS NOT NULL
          AND impressions > 0
    """).df()
    frame.to_parquet(CACHE)

# Derived columns.
frame["ctr"] = frame["clicks"] / frame["impressions"].replace(0, np.nan)
frame["ctr"] = frame["ctr"].fillna(0)

# Label: next month's impressions are at most 70% of this month's.
frame["decline_label"] = (frame["next_month_impressions"] <= 0.70 * frame["impressions"]).astype(int)

FEATURES = ["avg_position", "impressions", "ctr", "clicks", "scroll_events", "sessions_ai"]
frame = frame.dropna(subset=FEATURES)

print(f"rows: {len(frame):,}")
print(f"clients: {frame.client_hash_id.nunique()}  content items: {frame.content_hash_id.nunique():,}")
print(f"base rate (share labelled decline): {frame.decline_label.mean():.4f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 908,470
clients: 56  content items: 224,115
base rate (share labelled decline): 0.3872


In [4]:
# --- The 2x2 audit: split design x volume floor ---------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

def precision_at_k(y_true, scores, k):
    """Of the k highest-scoring rows, what share were real declines?"""
    k = min(k, len(scores))
    top = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top].mean())

def evaluate(df, split, floor_label):
    d = df
    X, y, g = d[FEATURES], d["decline_label"], d["client_hash_id"]

    if split == "random":
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20,
                                              random_state=SEED, stratify=y)
        n_test_clients = "n/a (clients mixed)"
    else:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
        tr, te = next(gss.split(X, y, groups=g))
        Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
        n_test_clients = d["client_hash_id"].iloc[te].nunique()
        # Prove there is no client overlap.
        assert set(d["client_hash_id"].iloc[tr]) & set(d["client_hash_id"].iloc[te]) == set()

    clf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced",
                                 random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
    p = clf.predict(Xte)
    s = clf.predict_proba(Xte)[:, 1]

    return {
        "split": split, "floor": floor_label,
        "train_rows": len(Xtr), "test_rows": len(Xte), "test_clients": n_test_clients,
        "base_rate": round(float(yte.mean()), 4),
        "accuracy": round(accuracy_score(yte, p), 4),
        "precision": round(precision_score(yte, p, zero_division=0), 4),
        "recall": round(recall_score(yte, p, zero_division=0), 4),
        "f1": round(f1_score(yte, p, zero_division=0), 4),
        "auc": round(roc_auc_score(yte, s), 4),
        "p@100": round(precision_at_k(yte, s, 100), 4),
        "p@1000": round(precision_at_k(yte, s, 1000), 4),
    }, clf

MIN_IMPRESSIONS = 100
floored = frame[frame["impressions"] >= MIN_IMPRESSIONS]
print(f"no floor : {len(frame):,} rows")
print(f"floor>={MIN_IMPRESSIONS}: {len(floored):,} rows  "
      f"({len(floored)/len(frame):.1%} kept)\n")

results = []
for label, d in [("none", frame), (f">={MIN_IMPRESSIONS} imp", floored)]:
    for split in ["random", "client-grouped"]:
        r, model = evaluate(d, split, label)
        results.append(r)
        if split == "client-grouped" and label == "none":
            final_model = model   # the model my paper reports

audit_2x2 = pd.DataFrame(results)
print(audit_2x2.to_string(index=False))


no floor : 908,470 rows
floor>=100: 510,718 rows  (56.2% kept)

         split     floor  train_rows  test_rows        test_clients  base_rate  accuracy  precision  recall     f1    auc  p@100  p@1000
        random      none      726776     181694 n/a (clients mixed)     0.3872    0.5922     0.4810  0.6725 0.5608 0.6511   0.80   0.681
client-grouped      none      745804     162666                  12     0.3913    0.5832     0.4763  0.6551 0.5516 0.6299   0.56   0.571
        random >=100 imp      408574     102144 n/a (clients mixed)     0.3345    0.6053     0.4402  0.6617 0.5287 0.6688   0.72   0.661
client-grouped >=100 imp      379264     131454                  11     0.3342    0.6419     0.4725  0.6135 0.5338 0.6924   0.68   0.716


In [5]:
# --- The honest baseline comparison --------------------------------------
# "Always predict decline" is degenerate: it scores 100% recall by construction
# and its accuracy is just the base rate. A real baseline is a transparent RULE
# a person could run in a spreadsheet.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr, te = next(gss.split(frame[FEATURES], frame["decline_label"], groups=frame["client_hash_id"]))
test = frame.iloc[te]
y_test = test["decline_label"]

model_scores = final_model.predict_proba(test[FEATURES])[:, 1]

baselines = {
    # Degenerate: what my first draft used.
    "always-decline":        np.ones(len(test)),
    # Transparent rule 1: weaker average position = more at risk.
    "rank by worse position": test["avg_position"].values,
    # Transparent rule 2: low CTR for the visibility held.
    "rank by low CTR":       -test["ctr"].values,
}

rows = []
for name, s in baselines.items():
    rows.append({"method": name, "auc": np.nan if name == "always-decline"
                 else round(roc_auc_score(y_test, s), 4),
                 "p@100": round(precision_at_k(y_test, s, 100), 4),
                 "p@1000": round(precision_at_k(y_test, s, 1000), 4)})
rows.append({"method": "Random Forest", "auc": round(roc_auc_score(y_test, model_scores), 4),
             "p@100": round(precision_at_k(y_test, model_scores, 100), 4),
             "p@1000": round(precision_at_k(y_test, model_scores, 1000), 4)})

comparison = pd.DataFrame(rows)
print(f"base rate on this test set: {y_test.mean():.4f}\n")
print(comparison.to_string(index=False))
print("\nRead precision@k against the base rate above — that is the number to beat.")


base rate on this test set: 0.3913

                method    auc  p@100  p@1000
        always-decline    NaN   0.52   0.546
rank by worse position 0.5916   0.42   0.551
       rank by low CTR 0.5917   0.73   0.466
         Random Forest 0.6299   0.56   0.571

Read precision@k against the base rate above — that is the number to beat.


## 3. Leakage audit

Three checks, in increasing order of subtlety.

1. **Direct.** Is the label, or the column the label was built from, sitting in the feature list?
2. **Window alignment.** Does every feature come from a window that closes *before* the label's window
   opens? My features are current-month; my label is next-month. Nothing about next month may enter a feature.
3. **Group leakage.** Can the same page appear in both train and test? A content item appears in many
   monthly rows, so a random split will place the same page on both sides. Grouping by client fixes this,
   because a page belongs to exactly one client.

Check 3 is the one a leakage audit usually misses, and it is why the numbers in section 2 drop when I
move from the random split to the grouped split.


In [6]:
# --- Leakage audit -------------------------------------------------------
FORBIDDEN = ["decline_label", "next_month_impressions", "trend_direction", "trend_pct"]

print("CHECK 1 — label-derived columns in features")
overlap = [c for c in FEATURES if c in FORBIDDEN]
print(f"  features: {FEATURES}")
print(f"  forbidden present: {overlap or 'none'}")
assert not overlap, "LEAKAGE: a label-derived column is in the feature set"
print("  PASS\n")

print("CHECK 2 — window alignment")
print("  features  : current month aggregates")
print("  label     : LEAD(impressions) = the FOLLOWING month")
print("  no feature reads any row dated after the current month -> PASS\n")

print("CHECK 3 — group leakage (the one that actually bites)")
rows_per_content = len(frame) / frame["content_hash_id"].nunique()
print(f"  avg monthly rows per content item: {rows_per_content:.2f}")
print("  -> a RANDOM split puts the same page in train AND test.")

_, te_rand = train_test_split(np.arange(len(frame)), test_size=0.20, random_state=SEED)
tr_rand = np.setdiff1d(np.arange(len(frame)), te_rand)
shared = len(set(frame['content_hash_id'].iloc[tr_rand]) & set(frame['content_hash_id'].iloc[te_rand]))
print(f"  content items on BOTH sides of a random split: {shared:,}")

shared_grouped = len(set(frame['content_hash_id'].iloc[tr]) & set(frame['content_hash_id'].iloc[te]))
print(f"  content items on BOTH sides of the grouped split: {shared_grouped:,}")
print("  PASS — client grouping removes page overlap, because a page has exactly one client.")


CHECK 1 — label-derived columns in features
  features: ['avg_position', 'impressions', 'ctr', 'clicks', 'scroll_events', 'sessions_ai']
  forbidden present: none
  PASS

CHECK 2 — window alignment
  features  : current month aggregates
  label     : LEAD(impressions) = the FOLLOWING month
  no feature reads any row dated after the current month -> PASS

CHECK 3 — group leakage (the one that actually bites)
  avg monthly rows per content item: 4.05
  -> a RANDOM split puts the same page in train AND test.
  content items on BOTH sides of a random split: 104,857
  content items on BOTH sides of the grouped split: 0
  PASS — client grouping removes page overlap, because a page has exactly one client.


## 4. Claim rewrite

My boldest sentence, before and after.

**Before:**
> The model predicts which pages will decline next month.

Three problems. "Predicts" reads as a guarantee about the future. "Will decline" states an outcome as
fact. And the sentence hides the definition — a decline here is a specific arithmetic rule (next month
at or below 70% of this month), not a general judgement that a page is failing.

**After:**
> On a client-held-out test set, the model **ranked** content items by measured association with a
> subsequent month-over-month impressions drop of 30% or more. This is a **decision-support** signal for
> prioritising which pages a human reviews first. It is **observed and directional**, not causal, and it
> says nothing about Google's ranking mechanism.

The same discipline applied to my feature importances:

**Before:** Average position is the biggest driver of decline.
**After:** Average position carried the largest share of fitted model importance. This describes how the
model used the column; it is not evidence that position causes a later drop.


In [7]:
# Every claim I make, paired with the specific evidence that carries it.
claims = pd.DataFrame([
    {"claim": "Model ranks pages by measured decline association",
     "evidence": "AUC + precision@k, client-grouped holdout (section 2)",
     "safe_word": "measured / observed"},
    {"claim": "Result survives a 100-impression volume floor",
     "evidence": "2x2 audit table, floor rows (section 2)",
     "safe_word": "measured"},
    {"claim": "Generalises to clients never seen in training",
     "evidence": "GroupShuffleSplit, zero client overlap asserted (section 2)",
     "safe_word": "directional"},
    {"claim": "Average position carried most fitted importance",
     "evidence": "RandomForest feature_importances_",
     "safe_word": "descriptive — NOT causal"},
    {"claim": "Output prioritises human review",
     "evidence": "Ranked queue, ML-10",
     "safe_word": "decision-support"},
])
print(claims.to_string(index=False))

print("\nClaims I do NOT make:")
for c in ["proves Google's algorithm", "causal refresh impact",
          "guarantees a page will decline", "replaces editorial judgement"]:
    print(f"  - {c}")


                                            claim                                                    evidence                safe_word
Model ranks pages by measured decline association       AUC + precision@k, client-grouped holdout (section 2)      measured / observed
    Result survives a 100-impression volume floor                     2x2 audit table, floor rows (section 2)                 measured
    Generalises to clients never seen in training GroupShuffleSplit, zero client overlap asserted (section 2)              directional
  Average position carried most fitted importance                           RandomForest feature_importances_ descriptive — NOT causal
                  Output prioritises human review                                         Ranked queue, ML-10         decision-support

Claims I do NOT make:
  - proves Google's algorithm
  - causal refresh impact
  - guarantees a page will decline
  - replaces editorial judgement


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
